# KLTN attack v2

Chọn **GPU T4 ×2**, bật Internet, sửa `DATASET_ROOT` và `NUM_SAMPLES`, rồi Run All. Mỗi GPU chạy một process riêng với đủ B16/B32/Laion; các cặp ảnh được chia round-robin. Kernel notebook không tải model lên GPU. Mặc định chạy 10 cặp, 300 bước, FP32, `epsilon=16`, `alpha=1`, `K=10`, giữ crop/loss/update FOA.

**Tiến độ:** một dòng `Attack: …%`, tính trên tổng số bước của mọi cặp ảnh trên cả hai GPU. Log pip/Git/model/K-means và traceback worker vào file, không tạo thanh progress lồng nhau. Lỗi thật báo ngắn ở notebook và có log chi tiết trong output.

**Tối ưu:** K-means `tqdm_flag=False`; bỏ nhánh text CLIP không dùng bởi `global_local_features`; lượt toàn ảnh chỉ phục vụ trạng thái/trọng số được chạy `no_grad` khi objective dùng crop. Vẫn thực hiện đủ hai lượt forward/loss theo thứ tự tác giả. Model loading được khóa lần lượt giữa worker để giảm peak RAM CPU; sau khi load xong, hai worker attack song song. Mỗi worker giới hạn CPU threads và prefetch cặp ảnh kế tiếp. Không bật AMP, không giảm số backbone/K/steps/token resolution.

**Protocol song song:** dynamic loss history được giữ **trong mỗi worker**, seed worker là `PAPER_SEED + rank`. Sharding đổi RNG và history so với v1.2 tuần tự; **không cam kết ảnh adversarial giống từng pixel với v1.2**. Config ghi seed, shard, source SHA, dependency, các tối ưu và timing để báo cáo thực nghiệm. `GPU_IDS=[0]`, `REQUIRE_TWO_GPUS=False` cho chế độ một GPU.

Mỗi run có thư mục riêng, manifest ghi global index và đúng source/target path; có thể đưa ZIP đã giải nén vào notebook captioning. Worker record files không dùng tên `manifest.jsonl` để captioning chỉ tìm thấy một manifest. Worker script `.py` được tạo lúc chạy từ các code cell bên dưới trong output, không cần upload script riêng. Sửa class/hàm thì chạy lại các cell implementation trước cell chạy attack.

Parity check dùng tensor nhỏ trên CPU, kiểm tra crop on/off và loss/gradient với code tác giả, không tải thêm model. Chi phí startup/download không được tính thành phần trăm attack; tiến độ giữ 0% trong giai đoạn đó. Không hứa tốc độ gấp đôi: đo timing trên Kaggle ở `summary.json`. Với ít nhất 2 cặp và đủ VRAM, cả hai GPU có công việc riêng.


In [ ]:
# Cell 1 — Cấu hình thực nghiệm và vận hành.
from pathlib import Path

DATASET_ROOT = Path('/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset')
WORK_ROOT = Path('/kaggle/working')
REPO_DIR = WORK_ROOT / 'FOA-Attack'
OUTPUT_ROOT = WORK_ROOT / 'foa_attack_outputs'
# Cache ngoài /kaggle/working để không xuất weights vào Kaggle Output.
HF_HOME = Path('/kaggle/temp/foa_hf_cache')
FOA_REPO_URL = 'https://github.com/jiaxiaojunQAQ/FOA-Attack.git'
KMEANS_REVISION = 'f7f36bd1cb4e3a761d73d584866d0a9c6b4d2805'

PAPER_NUM_SAMPLES = 100
PAPER_STEPS = 300
PAPER_EPSILON = 16.0
PAPER_ALPHA = 1.0
PAPER_INPUT_RES = 224
PAPER_CROP_SCALE = (0.5, 0.9)
PAPER_BACKBONES = ('B16', 'B32', 'Laion')
PAPER_CLUSTER_NUMBER = 10
PAPER_SEED = 2023

NUM_SAMPLES = 10       # None: tất cả cặp; 2: smoke test cả hai GPU.
STEPS = 300
GPU_IDS = [0, 1]       # Logical IDs trong CUDA_VISIBLE_DEVICES hiện tại.
REQUIRE_TWO_GPUS = True
USE_SOURCE_CROP = True
USE_TARGET_CROP = True
PRUNE_TEXT_BRANCH = True
NO_GRAD_FULL_IMAGE_PASS = True
CPU_THREADS_PER_WORKER = 2
PREFETCH_IMAGES = True
PROGRESS_WRITE_SECONDS = 1.0
POLL_SECONDS = 0.5
RUN_PARITY_CHECK = True
RUN_ATTACK = True
PREVIEW_OUTPUT = False


In [ ]:
# Cell 2 — Setup quiet: không đổi Torch/Torchvision của Kaggle.
import os, sys, json, ast, time, random, hashlib, shutil, subprocess
from datetime import datetime, timezone
from IPython import get_ipython
from IPython.display import display, HTML, FileLink

SETUP_DIR = WORK_ROOT / '_foa_setup_v13'
SETUP_DIR.mkdir(parents=True, exist_ok=True)

def quiet_command(command, log_name):
    log_path = SETUP_DIR / log_name
    with log_path.open('w', encoding='utf-8') as log:
        result = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT)
    if result.returncode:
        raise RuntimeError(f'Setup failed. Details: {log_path}')

if 'kmeans_pytorch' in sys.modules or 'surrogates' in sys.modules:
    raise RuntimeError('Restart Kaggle session before Run All after changing dependencies.')
PACKAGES = [
    'transformers==4.49.0', 'tokenizers==0.21.0', 'huggingface-hub==0.28.1',
    'ftfy==6.3.1', 'filelock==3.16.1', 'numba',
]
quiet_command([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES], 'pip.log')
quiet_command([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps',
               f'kmeans-pytorch @ git+https://github.com/subhadarship/kmeans_pytorch.git@{KMEANS_REVISION}'], 'kmeans_install.log')
os.environ.update({
    'HF_HOME': str(HF_HOME), 'HF_HUB_DISABLE_PROGRESS_BARS': '1',
    'TOKENIZERS_PARALLELISM': 'false', 'WANDB_MODE': 'disabled',
})
HF_HOME.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    quiet_command(['git', 'clone', '--depth', '1', FOA_REPO_URL, str(REPO_DIR)], 'git.log')
elif not (REPO_DIR / '.git').is_dir():
    raise RuntimeError(f'REPO_DIR is not a Git checkout: {REPO_DIR}')
else:
    origin = subprocess.check_output(['git', '-C', str(REPO_DIR), 'remote', 'get-url', 'origin'], text=True).strip()
    if origin.rstrip('/') != FOA_REPO_URL.rstrip('/'):
        raise RuntimeError('Existing REPO_DIR has a different origin.')
FOA_SOURCE_REVISION = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

WORKER_CODE = {}
def remember_worker_code(name):
    shell = get_ipython()
    if shell is None:
        raise RuntimeError('Run these implementation cells in a Jupyter/Kaggle notebook.')
    raw = shell.history_manager.input_hist_raw[-1]
    WORKER_CODE[name] = raw.split('# BEGIN WORKER CODE\n', 1)[1].split('# END WORKER CODE', 1)[0]


In [ ]:
# Cell 3 — Imports và kiểm tra topology; không tải weights trong kernel notebook.
import inspect, contextlib
from typing import Dict, List, Optional, Tuple, Any
from importlib import metadata
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from PIL import Image
import torch
from torch import nn, Tensor
import torch.nn.functional as F
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from torchvision.utils import save_image
from filelock import FileLock
from kmeans_pytorch import kmeans
from surrogates import (
    ClipB16FeatureExtractor, ClipB32FeatureExtractor, ClipLaionFeatureExtractor,
    EnsembleFeatureExtractor_ot, EnsembleFeatureLoss_OT_foa_attack,
)

if 'iter_limit' not in inspect.signature(kmeans).parameters:
    raise RuntimeError('Loaded kmeans lacks iter_limit; restart and rerun the pinned install.')
dist = metadata.distribution('kmeans-pytorch')
direct_url = json.loads(dist.read_text('direct_url.json') or '{}')
if direct_url.get('vcs_info', {}).get('commit_id') != KMEANS_REVISION:
    raise RuntimeError('kmeans distribution is not the pinned Git revision.')
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f'Add Input and correct DATASET_ROOT: {DATASET_ROOT}')
if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU T4 x2 in Kaggle Settings.')
if not GPU_IDS or len(set(GPU_IDS)) != len(GPU_IDS) or any(type(i) is not int or i < 0 or i >= torch.cuda.device_count() for i in GPU_IDS):
    raise ValueError('GPU_IDS must contain distinct available logical GPU indices.')
if REQUIRE_TWO_GPUS and len(GPU_IDS) != 2:
    raise RuntimeError('Select Kaggle GPU T4 x2, or set GPU_IDS=[0] and REQUIRE_TWO_GPUS=False.')
if type(STEPS) is not int or STEPS <= 0 or (NUM_SAMPLES is not None and (type(NUM_SAMPLES) is not int or NUM_SAMPLES <= 0)):
    raise ValueError('STEPS/NUM_SAMPLES must be positive integers; NUM_SAMPLES may be None.')
if CPU_THREADS_PER_WORKER < 1 or POLL_SECONDS <= 0 or PROGRESS_WRITE_SECONDS <= 0:
    raise ValueError('Thread count and polling intervals must be positive.')

# Do not print device/model reprs or download progress; metadata is written to run_config.json.
GPU_METADATA = [{'logical_id': i, 'name': torch.cuda.get_device_name(i),
                 'memory_bytes': torch.cuda.get_device_properties(i).total_memory} for i in GPU_IDS]


In [ ]:
# Worker implementation: preprocessing
# BEGIN WORKER CODE
# Cell 4 — Dataset discovery and preprocessing (editable research boundary).
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def count_images(path: Path) -> int:
    """Return recursive number of supported image files below ``path``."""
    return sum(p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS for p in path.rglob('*'))

def find_imagefolder(root: Path, expected_name: str) -> Path:
    """Find the largest ImageFolder named ``expected_name`` in a Kaggle dataset."""
    candidates = sorted((p for p in root.rglob(expected_name) if p.is_dir()), key=lambda p: -count_images(p))
    if not candidates: raise FileNotFoundError(f'Cannot find {expected_name} below {root}')
    return candidates[0]

def pil_to_foa_tensor(image: Image.Image) -> Tensor:
    """Convert PIL RGB image to the original FOA tensor in pixel range [0, 255]."""
    array = np.array(image, np.uint8, copy=True)
    return torch.from_numpy(array).permute(2, 0, 1).contiguous().to(torch.get_default_dtype())


# Preprocessing
def preprocess_image(input_res: int = PAPER_INPUT_RES) -> transforms.Compose:
    """Build the exact Resize → CenterCrop → RGB → pixel-tensor preprocessing of FOA."""
    return transforms.Compose([
        transforms.Resize(input_res, interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(input_res), transforms.Lambda(lambda x: x.convert('RGB')),
        transforms.Lambda(pil_to_foa_tensor),
    ])

class ImageFolderWithPaths(datasets.ImageFolder):
    """ImageFolder that additionally returns the source image path for manifest export."""
    def __getitem__(self, index: int):
        """Return transformed image, class label, and original filesystem path."""
        image, label = super().__getitem__(index)
        return image, label, self.samples[index][0]


# END WORKER CODE
remember_worker_code('preprocessing')


In [ ]:
# Worker implementation: models
# BEGIN WORKER CODE
BACKBONE_MAP = {'B16': ClipB16FeatureExtractor, 'B32': ClipB32FeatureExtractor, 'Laion': ClipLaionFeatureExtractor}

def set_foa_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class QuietEnsembleFeatureExtractor(EnsembleFeatureExtractor_ot):
    """Keep author's forward/K-means defaults; disable logging inside clustering itself."""
    def get_cluster_center(self, embedding_img, device):
        _, centers = kmeans(X=embedding_img, num_clusters=self.cluster_number,
                            distance='euclidean', device=device, tqdm_flag=False)
        if not torch.isfinite(centers).all():
            raise FloatingPointError('Non-finite source K-means centers.')
        return centers.to(device)

def build_foa_ensemble(device, backbones, cluster_number, prune_text):
    models = []
    for name in backbones:
        model = BACKBONE_MAP[name]()
        # global_local_features only accesses CLIP vision_model, never these text modules.
        # Remove before .to(device) to avoid allocating unused text weights on T4.
        if prune_text:
            for attr in ('text_model', 'text_projection'):
                if hasattr(model.model, attr):
                    delattr(model.model, attr)
        model = model.eval().requires_grad_(False).to(device)
        models.append(model)
    return QuietEnsembleFeatureExtractor(models, cluster_number=cluster_number), models

# END WORKER CODE
remember_worker_code('models')


In [ ]:
# Worker implementation: loss
# BEGIN WORKER CODE
# Cell 6 — FOA loss. This is the paper loss copied into an editable cell.
class FOAOptimalTransportLoss(nn.Module):
    """Original FOA global/local OT loss with its dynamic ensemble weighting (K=10 by default)."""
    def __init__(self, extractors: List[nn.Module], cluster_number: int = PAPER_CLUSTER_NUMBER):
        """Initialize FOA state for the frozen surrogate extractors and local cluster count."""
        super().__init__(); self.extractors = nn.ModuleList(extractors); self.cluster_number = cluster_number
        self.ground_truth, self.ground_truth_local, self.previous_loss_list = [], [], []

    @torch.no_grad()
    def set_ground_truth(self, x: Tensor) -> None:
        """Extract target global features and K-means local centers as FOA ground truth."""
        self.ground_truth.clear(); self.ground_truth_local.clear()
        for model in self.extractors:
            global_feature, local_feature = model.global_local_features(x.to(x.device))
            centers = self.cluster_centers(local_feature.squeeze(0), x.device).unsqueeze(0)
            self.ground_truth.append(global_feature); self.ground_truth_local.append(centers)

    def cluster_centers(self, embeddings: Tensor, device: torch.device) -> Tensor:
        """Compute original Euclidean K-means local centers (100 iteration limit)."""
        _, centers = kmeans(X=embeddings, num_clusters=self.cluster_number, distance='euclidean', device=device, iter_limit=100, tqdm_flag=False)
        if not torch.isfinite(centers).all(): raise FloatingPointError('Non-finite target K-means centers.')
        return centers.to(device)

    def optimal_transport(self, source: Tensor, target: Tensor) -> Tensor:
        """Return FOA cosine-similarity optimal-transport score using original Sinkhorn settings."""
        similarity = torch.einsum('md,nd->mn', F.normalize(source, dim=1), F.normalize(target, dim=1)).contiguous()
        u = torch.full((source.shape[0],), 1 / source.shape[0], dtype=similarity.dtype, device=similarity.device)
        v = torch.full((target.shape[0],), 1 / target.shape[0], dtype=similarity.dtype, device=similarity.device)
        with torch.no_grad():
            kernel = torch.exp(-(1 - similarity) / 0.1)
            transport = self.sinkhorn(kernel, u, v)
        if not torch.isfinite(transport).all(): raise FloatingPointError('Non-finite FOA transport.')
        return torch.sum(transport * similarity)

    def sinkhorn(self, kernel: Tensor, u: Tensor, v: Tensor) -> Tensor:
        """Run the original 100-step, 1e-2-threshold Sinkhorn normalization."""
        r, c = torch.ones_like(u), torch.ones_like(v)
        for _ in range(100):
            old_r = r; r = u / (kernel @ c.unsqueeze(-1)).squeeze(-1); c = v / (kernel.T @ r.unsqueeze(-1)).squeeze(-1)
            if (r - old_r).abs().mean().item() < 1e-2: break
        return torch.outer(r, c) * kernel

    def forward(self, global_features: Dict[int, Tensor], local_features: Dict[int, Tensor]) -> Tensor:
        """Compute original FOA global OT + 0.2 local OT and dynamic model weights."""
        losses = [self.optimal_transport(self.ground_truth[i], global_features[i].unsqueeze(0)) + 0.2 * self.optimal_transport(self.ground_truth_local[i].squeeze(0), local_features[i].squeeze(0)) for i in range(len(self.extractors))]
        if not self.previous_loss_list: self.previous_loss_list = [loss.detach() for loss in losses]
        weights = np.array([loss.item() / (prev.item() + 1e-8) for loss, prev in zip(losses, self.previous_loss_list)])
        with np.errstate(over='ignore', invalid='ignore'):
            weights = np.exp(weights) / np.exp(weights).sum() * len(weights)
        if not np.isfinite(weights).all(): raise FloatingPointError('FOA dynamic weights overflowed.')
        self.previous_loss_list = [loss.detach() for loss in losses]
        return sum(float(weights[i]) * loss for i, loss in enumerate(losses))


# END WORKER CODE
remember_worker_code('loss')


In [ ]:
# Worker implementation: attack
# BEGIN WORKER CODE
def foa_fgsm_attack(image_org, image_tgt, extractor, loss_fn, source_crop, target_crop,
                    steps, epsilon, alpha, on_step=None, no_grad_full=True):
    """Same FGSM/crop objective; eliminate unused full-image autograd graph and nested tqdm."""
    if image_org.shape != image_tgt.shape or image_org.ndim != 4 or image_org.shape[0] != 1:
        raise ValueError('Original FOA extractor supports one equally sized image pair per batch.')
    if not torch.isfinite(image_org).all() or not torch.isfinite(image_tgt).all():
        raise FloatingPointError('Non-finite attack input.')
    delta = torch.zeros_like(image_org, requires_grad=True)
    source_is_identity = isinstance(source_crop, nn.Identity)
    for _ in range(steps):
        with torch.no_grad():
            loss_fn.set_ground_truth(target_crop(image_tgt))
        adv_image = image_org + delta
        # This forward still updates dynamic weights; do not skip it when source crop is enabled.
        context = torch.no_grad() if no_grad_full and not source_is_identity else contextlib.nullcontext()
        with context:
            global_features, local_features = extractor(adv_image)
            global_score = loss_fn(global_features, local_features)
        if source_is_identity:
            objective = global_score
        else:
            del global_features, local_features, global_score
            crop_global, crop_local = extractor(source_crop(adv_image))
            objective = loss_fn(crop_global, crop_local)
        if not torch.isfinite(objective).all():
            raise FloatingPointError('Non-finite FOA objective.')
        gradient = torch.autograd.grad(objective, delta, create_graph=False)[0]
        if not torch.isfinite(gradient).all():
            raise FloatingPointError('Non-finite FOA gradient.')
        delta.data = torch.clamp(delta + alpha * torch.sign(gradient), min=-epsilon, max=epsilon)
        if on_step is not None:
            on_step()
    return torch.clamp((image_org + delta) / 255.0, 0.0, 1.0).detach()

# END WORKER CODE
remember_worker_code('attack')


In [ ]:
# Worker implementation: worker_runtime
# BEGIN WORKER CODE
def atomic_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(payload, ensure_ascii=False), encoding='utf-8')
    for attempt in range(6):
        try:
            os.replace(temporary, path)
            break
        except PermissionError:
            if os.name != 'nt' or attempt == 5:
                raise
            time.sleep(0.02)  # Windows reader sharing; Kaggle/Linux replaces atomically.

class WorkerProgress:
    def __init__(self, path, rank, total_steps, interval):
        self.path, self.rank, self.total_steps = Path(path), rank, total_steps
        self.interval, self.completed, self.last_write = interval, 0, float('-inf')
        self.phase, self.current_index = 'loading', None
    def write(self, force=False, **extra):
        now = time.monotonic()
        if force or now - self.last_write >= self.interval:
            atomic_json(self.path, {'rank': self.rank, 'phase': self.phase, 'completed_steps': self.completed,
                                  'total_steps': self.total_steps, 'sample_index': self.current_index, **extra})
            self.last_write = now
    def step(self):
        self.completed += 1
        self.write()

def load_pair(pair, transform, pin_memory):
    with Image.open(pair['source_path']) as image:
        source = transform(image).unsqueeze(0)
    with Image.open(pair['target_path']) as image:
        target = transform(image).unsqueeze(0)
    if pin_memory:
        source, target = source.pin_memory(), target.pin_memory()
    return source, target

def worker_main(config):
    rank, pairs = config['rank'], config['pairs']
    output = Path(config['output_dir'])
    progress = WorkerProgress(config['progress_path'], rank, len(pairs) * config['steps'], config['progress_write_seconds'])
    progress.write(force=True)
    started = time.perf_counter()
    completed_samples = 0
    try:
        device = torch.device(config['device'])
        if device.type == 'cuda':
            torch.cuda.set_device(device)
        torch.set_num_threads(config['cpu_threads'])
        torch.set_num_interop_threads(1)
        set_foa_seed(config['worker_seed'])
        # Serial model loading avoids two large LAION CPU state dictionaries peaking together.
        with FileLock(str(output / 'model_load.lock')):
            extractor, models = build_foa_ensemble(device, tuple(config['backbones']), config['cluster_number'], config['prune_text'])
        if device.type == 'cuda':
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats(device)
        loaded_at = time.perf_counter()
        loss_fn = FOAOptimalTransportLoss(models, cluster_number=config['cluster_number'])
        source_crop = transforms.RandomResizedCrop(config['input_res'], scale=tuple(config['crop_scale'])) if config['use_source_crop'] else nn.Identity()
        target_crop = transforms.RandomResizedCrop(config['input_res'], scale=tuple(config['crop_scale'])) if config['use_target_crop'] else nn.Identity()
        transform = preprocess_image(config['input_res'])
        timings = []
        progress.phase = 'attack'
        progress.write(force=True)
        # Preprocessing is deterministic; one prefetch thread overlaps CPU image loading with attack.
        with ThreadPoolExecutor(max_workers=1) as prefetch, Path(config['records_path']).open('w', encoding='utf-8') as records:
            future = prefetch.submit(load_pair, pairs[0], transform, device.type == 'cuda') if config['prefetch'] and pairs else None
            for position, pair in enumerate(pairs):
                progress.current_index = pair['index']
                sample_started = time.perf_counter()
                source, target = future.result() if future is not None else load_pair(pair, transform, device.type == 'cuda')
                future = prefetch.submit(load_pair, pairs[position + 1], transform, device.type == 'cuda') if config['prefetch'] and position + 1 < len(pairs) else None
                source = source.to(device, non_blocking=True)
                target = target.to(device, non_blocking=True)
                adversarial = foa_fgsm_attack(source, target, extractor, loss_fn, source_crop, target_crop,
                                              steps=config['steps'], epsilon=config['epsilon'], alpha=config['alpha'],
                                              on_step=progress.step, no_grad_full=config['no_grad_full'])
                if not torch.isfinite(adversarial).all():
                    raise FloatingPointError('Refusing to save non-finite image.')
                stem = f"{pair['index']:05d}"
                paths = {role: f'images/{stem}_{role}.png' for role in ('source', 'target', 'adversarial')}
                save_image(source / 255.0, output / paths['source'])
                save_image(target / 255.0, output / paths['target'])
                save_image(adversarial, output / paths['adversarial'])
                saved = np.asarray(Image.open(output / paths['adversarial'])).astype(np.int16)
                original = np.asarray(Image.open(output / paths['source'])).astype(np.int16)
                linf = int(np.abs(saved - original).max())
                if linf > config['epsilon']:
                    raise RuntimeError(f'PNG violates L_inf budget: {linf}')
                record = {'index': pair['index'], 'sample_id': stem, 'worker_rank': rank,
                          'original_path': pair['source_path'], 'target_path': pair['target_path'],
                          'linf_255': linf, **paths}
                records.write(json.dumps(record) + '\n')
                records.flush()
                completed_samples += 1
                timings.append({'index': pair['index'], 'seconds': time.perf_counter() - sample_started})
                progress.write(force=True)
                del source, target, adversarial
        if device.type == 'cuda':
            torch.cuda.synchronize(device)
        atomic_json(config['timing_path'], {
            'rank': rank, 'worker_seed': config['worker_seed'], 'samples': completed_samples,
            'startup_seconds': loaded_at - started, 'attack_seconds': time.perf_counter() - loaded_at,
            'total_seconds': time.perf_counter() - started, 'per_sample': timings,
            'peak_allocated_bytes': torch.cuda.max_memory_allocated(device) if device.type == 'cuda' else None,
            'peak_reserved_bytes': torch.cuda.max_memory_reserved(device) if device.type == 'cuda' else None,
        })
        progress.phase = 'complete'
        progress.write(force=True, completed_samples=completed_samples)
    except BaseException as error:
        progress.phase = 'error'
        progress.write(force=True, completed_samples=completed_samples, error_type=type(error).__name__, error=str(error))
        raise

# END WORKER CODE
remember_worker_code('worker_runtime')


In [ ]:
# Cell 9 — Validate the loss AND both crop branches against the original implementation.
def assert_paper_defaults() -> None:
    """Check reference constants, while allowing STEPS/NUM_SAMPLES experiment overrides."""
    assert (PAPER_EPSILON, PAPER_ALPHA, PAPER_STEPS, PAPER_CLUSTER_NUMBER) == (16.0, 1.0, 300, 10)
    assert PAPER_BACKBONES == ('B16', 'B32', 'Laion') and PAPER_CROP_SCALE == (0.5, 0.9)


def verify_update_parity() -> None:
    """Compare real loss/gradients and FGSM branch behavior without loading extra models."""
    assert_paper_defaults()
    import ast
    from types import SimpleNamespace
    source = (REPO_DIR / 'generate_adversarial_samples_foa_attack.py').read_text(encoding='utf-8')
    original_node = next(node for node in ast.parse(source).body if isinstance(node, ast.FunctionDef) and node.name == 'fgsm_attack')
    namespace = {'torch': torch, 'nn': nn, 'tqdm': lambda sequence, **_: sequence,
                 'log_metrics': lambda *_, **__: None, 'MainConfig': object,
                 'Optional': Optional, 'transforms': transforms}
    exec(compile(ast.Module(body=[original_node], type_ignores=[]), '<reference-foa>', 'exec'), namespace)

    class MockExtractor(nn.Module):
        def forward(self, x):
            return {0: x.mean((2, 3)).squeeze(0)}, {0: x.mean((2, 3)).unsqueeze(1)}

    class MockLoss(nn.Module):
        def __init__(self):
            super().__init__()
            self.trace = []
        def set_ground_truth(self, x):
            self.target = x.mean((2, 3)).detach()
            self.trace.append(('target', tuple(x.shape)))
        def forward(self, global_f, local_f):
            self.trace.append(('score', global_f[0].detach().cpu().tolist()))
            return -(global_f[0] - self.target.squeeze(0)).square().mean()

    class MockCrop(nn.Module):
        def forward(self, x):
            return x[..., :3, :3]

    # Includes pixel clipping boundaries and a non-uniform input so crop is observable.
    x = torch.linspace(0, 255, 48, device='cpu').reshape(1, 3, 4, 4)
    y = torch.clamp(x + 30, 0, 255)
    for crop_enabled in (False, True):
        source_crop = MockCrop() if crop_enabled else nn.Identity()
        target_crop = MockCrop() if crop_enabled else nn.Identity()
        ours_loss, original_loss = MockLoss(), MockLoss()
        ours = foa_fgsm_attack(x, y, MockExtractor(), ours_loss, source_crop, target_crop,
                               steps=20, epsilon=16, alpha=1, no_grad_full=NO_GRAD_FULL_IMAGE_PASS)
        cfg = SimpleNamespace(optim=SimpleNamespace(steps=20, epsilon=16, alpha=1, no_grad_full=NO_GRAD_FULL_IMAGE_PASS),
                              model=SimpleNamespace(use_source_crop=crop_enabled))
        original = namespace['fgsm_attack'](cfg, MockExtractor(), original_loss, source_crop, target_crop, 0, x, y)
        torch.testing.assert_close(ours, original)
        assert ours_loss.trace == original_loss.trace, 'Global/crop loss evaluation order differs from the author.'
        assert torch.max(torch.abs(ours * 255 - x)).item() <= 16 + 1e-4

    # Compare actual OT/dynamic-weight values and input gradients on several successive forwards.
    # Use a private Generator so this guard cannot perturb K-means/crop RNG in the experiment.
    rng = torch.Generator(device='cpu').manual_seed(927)
    models = [nn.Identity() for _ in range(3)]
    ours_loss = FOAOptimalTransportLoss(models)
    original_loss = EnsembleFeatureLoss_OT_foa_attack(models, cluster_number=PAPER_CLUSTER_NUMBER)
    for i in range(3):
        global_target = F.normalize(torch.randn(1, 8 + i, generator=rng), dim=-1).to('cpu')
        local_target = F.normalize(torch.randn(1, 10, 8 + i, generator=rng), dim=-1).to('cpu')
        for loss in (ours_loss, original_loss):
            loss.ground_truth.append(global_target.clone())
            loss.ground_truth_local.append(local_target.clone())
    for _ in range(3):
        global_features = {i: F.normalize(torch.randn(8 + i, generator=rng), dim=-1).to('cpu').requires_grad_() for i in range(3)}
        local_features = {i: F.normalize(torch.randn(1, 10, 8 + i, generator=rng), dim=-1).to('cpu').requires_grad_() for i in range(3)}
        original_global = {i: value.detach().clone().requires_grad_() for i, value in global_features.items()}
        original_local = {i: value.detach().clone().requires_grad_() for i, value in local_features.items()}
        ours_score = ours_loss(global_features, local_features)
        original_score = original_loss(original_global, original_local)
        torch.testing.assert_close(ours_score, original_score, rtol=1e-5, atol=1e-6)
        ours_grad = torch.autograd.grad(ours_score, tuple(global_features.values()) + tuple(local_features.values()))
        original_grad = torch.autograd.grad(original_score, tuple(original_global.values()) + tuple(original_local.values()))
        for actual, expected in zip(ours_grad, original_grad):
            torch.testing.assert_close(actual, expected, rtol=1e-5, atol=1e-6)



if RUN_PARITY_CHECK:
    verify_update_parity()


In [ ]:
# Cell 10 — Generate standalone worker program from the exact implementation cells just executed.
WORKER_HEADER = """import os, sys, json, time, random, contextlib, traceback
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
from concurrent.futures import ThreadPoolExecutor
from importlib import metadata
CONFIG = json.loads(Path(sys.argv[1]).read_text(encoding='utf-8'))
sys.path.insert(0, CONFIG['repo_dir'])
import numpy as np
from PIL import Image
import torch
from torch import nn, Tensor
import torch.nn.functional as F
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from torchvision.utils import save_image
from filelock import FileLock
from kmeans_pytorch import kmeans
from surrogates import ClipB16FeatureExtractor, ClipB32FeatureExtractor, ClipLaionFeatureExtractor, EnsembleFeatureExtractor_ot
PAPER_SEED = CONFIG['paper_seed']
PAPER_INPUT_RES = CONFIG['input_res']
PAPER_CLUSTER_NUMBER = CONFIG['cluster_number']
"""
WORKER_ENTRY = """
if __name__ == '__main__':
    try:
        worker_main(CONFIG)
    except BaseException:
        traceback.print_exc()
        sys.exit(1)
"""
expected = ['preprocessing', 'models', 'loss', 'attack', 'worker_runtime']
if any(name not in WORKER_CODE for name in expected):
    raise RuntimeError('Execute all worker implementation cells before launching the attack.')
WORKER_PROGRAM = WORKER_HEADER + '\n'.join(WORKER_CODE[name] for name in expected) + WORKER_ENTRY
ast.parse(WORKER_PROGRAM)


In [ ]:
# Cell 11 — Scheduler: one fresh Python process per GPU, one percentage display.
def read_worker_progress(path):
    try:
        return json.loads(Path(path).read_text(encoding='utf-8'))
    except (FileNotFoundError, PermissionError, json.JSONDecodeError):
        return {}

def collect_records(jobs):
    result = {}
    for job in jobs:
        path = Path(job['records_path'])
        if not path.exists():
            continue
        for line in path.read_text(encoding='utf-8').splitlines():
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue  # Worker killed during its final write; retain complete earlier records.
            index = record['index']
            if index in result:
                raise RuntimeError(f'Duplicate sample index from workers: {index}')
            result[index] = record
    return [result[index] for index in sorted(result)]

def monitor_workers(jobs, worker_program, run_dir, environments, poll_seconds, update_percent):
    processes, logs = [], []
    total_steps = sum(len(job['pairs']) * job['steps'] for job in jobs)
    started = time.perf_counter()
    last_percentage = None
    status = {'status': 'running', 'samples': 0, 'requested_samples': sum(len(job['pairs']) for job in jobs)}
    error = None
    try:
        for job, environment in zip(jobs, environments):
            job_path = run_dir / 'jobs' / f"gpu_{job['rank']}.json"
            job_path.write_text(json.dumps(job), encoding='utf-8')
            log = (run_dir / 'logs' / f"gpu_{job['rank']}.log").open('w', encoding='utf-8')
            logs.append(log)
            processes.append(subprocess.Popen([sys.executable, '-u', str(worker_program), str(job_path)],
                                               env=environment, stdout=log, stderr=subprocess.STDOUT))
        while True:
            snapshots = [read_worker_progress(job['progress_path']) for job in jobs]
            completed = sum(min(s.get('completed_steps', 0), len(job['pairs']) * job['steps']) for s, job in zip(snapshots, jobs))
            percent = min(99.9, 100.0 * completed / total_steps)
            rounded = round(percent, 1)
            if rounded != last_percentage:
                update_percent(rounded)
                last_percentage = rounded
            failed = [i for i, process in enumerate(processes) if process.poll() not in (None, 0)]
            if failed:
                raise RuntimeError(f"Worker {failed} failed. Details: {run_dir / 'logs'}")
            if all(process.poll() is not None for process in processes):
                break
            time.sleep(poll_seconds)
        records = collect_records(jobs)
        expected = {pair['index'] for job in jobs for pair in job['pairs']}
        if {record['index'] for record in records} != expected:
            raise RuntimeError('Workers exited without all expected samples; inspect worker logs.')
        status['status'] = 'complete'
    except BaseException as caught:
        error = caught
        status['status'] = 'interrupted' if isinstance(caught, KeyboardInterrupt) else 'error'
        status['error_type'] = type(caught).__name__
        status['error'] = str(caught)
    finally:
        for process in processes:
            if process.poll() is None:
                process.terminate()
        for process in processes:
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        for log in logs:
            log.close()
        records = collect_records(jobs)
        (run_dir / 'manifest.jsonl').write_text(''.join(json.dumps(record) + '\n' for record in records), encoding='utf-8')
        status.update(samples=len(records), wall_seconds=time.perf_counter() - started)
        atomic_json(run_dir / 'status.json', status)
        timings = [read_worker_progress(job['timing_path']) for job in jobs]
        atomic_json(run_dir / 'summary.json', {
            **status, 'workers': timings, 'samples_per_second_including_startup': len(records) / status['wall_seconds'],
            'protocol': 'foa_parallel_worker_history_v1_3',
        })
    if error is not None:
        raise error
    update_percent(100.0)
    return records

RUN_DIR = None
if RUN_ATTACK:
    source_dir = find_imagefolder(DATASET_ROOT, 'bigscale')
    target_dir = find_imagefolder(DATASET_ROOT, 'target_images')
    source_paths = datasets.ImageFolder(source_dir).samples
    target_paths = datasets.ImageFolder(target_dir).samples
    available = min(len(source_paths), len(target_paths))
    requested = available if NUM_SAMPLES is None else NUM_SAMPLES
    if requested <= 0 or requested > available:
        raise ValueError(f'Requested {requested} samples; only {available} paired samples available.')
    if REQUIRE_TWO_GPUS and requested < 2:
        raise ValueError('Use at least NUM_SAMPLES=2 to give both GPUs work.')
    active_ids = GPU_IDS[:min(len(GPU_IDS), requested)]
    pairs = [{'index': index, 'source_path': str(Path(source_paths[index][0]).resolve()),
              'target_path': str(Path(target_paths[index][0]).resolve())} for index in range(requested)]
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    RUN_DIR = OUTPUT_ROOT / f'kltn_attack_v1_3_{stamp}'
    for folder in ('images', 'jobs', 'logs', 'progress', 'records', 'timings'):
        (RUN_DIR / folder).mkdir(parents=True, exist_ok=False)
    worker_file = RUN_DIR / 'worker_program.py'
    worker_file.write_text(WORKER_PROGRAM, encoding='utf-8')
    visible = os.environ.get('CUDA_VISIBLE_DEVICES')
    visible_ids = [token.strip() for token in visible.split(',')] if visible else None
    jobs, environments = [], []
    for rank, logical_id in enumerate(active_ids):
        visibility = visible_ids[logical_id] if visible_ids is not None else str(logical_id)
        job = {
            'rank': rank, 'gpu_logical_id': logical_id, 'cuda_visibility': visibility, 'device': 'cuda:0',
            'repo_dir': str(REPO_DIR.resolve()), 'output_dir': str(RUN_DIR.resolve()),
            'pairs': pairs[rank::len(active_ids)], 'backbones': list(PAPER_BACKBONES),
            'paper_seed': PAPER_SEED, 'worker_seed': PAPER_SEED + rank,
            'steps': STEPS, 'epsilon': PAPER_EPSILON, 'alpha': PAPER_ALPHA,
            'input_res': PAPER_INPUT_RES, 'crop_scale': list(PAPER_CROP_SCALE), 'cluster_number': PAPER_CLUSTER_NUMBER,
            'use_source_crop': USE_SOURCE_CROP, 'use_target_crop': USE_TARGET_CROP,
            'prune_text': PRUNE_TEXT_BRANCH, 'no_grad_full': NO_GRAD_FULL_IMAGE_PASS,
            'cpu_threads': CPU_THREADS_PER_WORKER, 'prefetch': PREFETCH_IMAGES,
            'progress_write_seconds': PROGRESS_WRITE_SECONDS,
            'progress_path': str((RUN_DIR / 'progress' / f'gpu_{rank}.json').resolve()),
            'records_path': str((RUN_DIR / 'records' / f'gpu_{rank}.jsonl').resolve()),
            'timing_path': str((RUN_DIR / 'timings' / f'gpu_{rank}.json').resolve()),
        }
        environment = os.environ.copy()
        environment.update({'CUDA_VISIBLE_DEVICES': visibility, 'PYTHONHASHSEED': str(job['worker_seed']),
                            'OMP_NUM_THREADS': str(CPU_THREADS_PER_WORKER), 'MKL_NUM_THREADS': str(CPU_THREADS_PER_WORKER),
                            'OPENBLAS_NUM_THREADS': str(CPU_THREADS_PER_WORKER), 'PYTHONUNBUFFERED': '1'})
        jobs.append(job)
        environments.append(environment)
    config = {
        'notebook': 'kltn-attack-v1.3.ipynb', 'protocol': 'foa_parallel_worker_history_v1_3',
        'foa_source_revision': FOA_SOURCE_REVISION, 'kmeans_revision': KMEANS_REVISION,
        'worker_program_sha256': hashlib.sha256(WORKER_PROGRAM.encode()).hexdigest(),
        'packages': PACKAGES, 'torch': torch.__version__, 'gpus': GPU_METADATA,
        'num_samples': requested, 'steps': STEPS, 'epsilon': PAPER_EPSILON, 'alpha': PAPER_ALPHA,
        'jobs': jobs, 'sharding': 'round_robin', 'loss_history': 'persistent within each worker',
        'sequential_v12_pixel_parity': False,
    }
    (RUN_DIR / 'run_config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
    progress_display = display(HTML('<div>Attack: 0.0%</div>'), display_id=True)
    def update_attack_percentage(percent):
        progress_display.update(HTML(f'<div>Attack: {percent:.1f}%</div>'))
    completed_records = monitor_workers(jobs, worker_file, RUN_DIR, environments, POLL_SECONDS, update_attack_percentage)


In [ ]:
# Cell 12 — ZIP/download; cũng xuất partial artifacts nếu cell trước bị ngắt/lỗi.
if RUN_DIR is not None and RUN_DIR.is_dir():
    if PREVIEW_OUTPUT and (RUN_DIR / 'manifest.jsonl').exists():
        records = [json.loads(line) for line in (RUN_DIR / 'manifest.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
        if records:
            import matplotlib.pyplot as plt
            fig, axes = plt.subplots(1, 3, figsize=(14, 5))
            for axis, role in zip(axes, ('source', 'target', 'adversarial')):
                axis.imshow(Image.open(RUN_DIR / records[0][role]))
                axis.set_title(role)
                axis.axis('off')
            plt.show()
    archive = Path(shutil.make_archive(str(WORK_ROOT / RUN_DIR.name), 'zip', root_dir=RUN_DIR))
    display(FileLink(str(archive)))
